# AutoE2E in the AlpaSim closed loop: a walkthrough

This notebook follows AutoE2E through one closed-loop rollout in NVIDIA AlpaSim, using the driver plugin in
`Model/plugins/alpasim_driver/`. It ends with the reward signal that closed-loop fine-tuning is meant to train on.
Read it next to the [overview](overview.md).

| Part | What it shows | Status |
| :--- | :--- | :--- |
| 1–3 | Setup, the policy checkpoint, and the camera rig the driver loads | implemented |
| 4 | One driving decision, offline: AlpaSim input, model inputs, controls, waypoints | implemented |
| 5 | Launching a closed-loop rollout (off by default) | implemented |
| 6–7 | What happened in a rollout, and its phase-1 reward | implemented |
| 8 | The per-decision table an RL update would consume | the RL update itself is **planned**, not implemented |

**Prerequisites** (see the overview's "Run it yourself"):

- an AlpaSim checkout with the plugin installed; use its `.venv` as the kernel
- a released AutoE2E checkpoint; this notebook uses nuPlan Epoch 5
- for parts 6–8, a finished rollout of `driver=autoe2e_nuplan`

Paths come from the environment variables `ALPASIM_ROOT`, `AUTOE2E_CHECKPOINT` and `ROLLOUT_LOG_DIR`. Their defaults
follow the layout in the plugin README.

## 1. Setup

In [ ]:
import collections
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml


def find_repo(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "Model" / "plugins" / "alpasim_driver").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the auto_e2e repository, or set AUTOE2E_REPO.")


REPO = Path(os.environ["AUTOE2E_REPO"]) if "AUTOE2E_REPO" in os.environ else find_repo(Path.cwd().resolve())
PLUGIN_DIR = REPO / "Model" / "plugins" / "alpasim_driver"
CONFIG_DIR = PLUGIN_DIR / "alpasim_autoe2e" / "configs" / "driver"
ALPASIM_ROOT = Path(os.environ.get("ALPASIM_ROOT", REPO.parent / "alpasim"))
CHECKPOINT = Path(
    os.environ.get("AUTOE2E_CHECKPOINT", ALPASIM_ROOT / "data" / "drivers" / "autoe2e" / "checkpoint.pt")
)
ROLLOUT_LOG_DIR = Path(os.environ.get("ROLLOUT_LOG_DIR", ALPASIM_ROOT / "outputs" / "autoe2e_closed_loop_run"))
RUN_SIM = os.environ.get("RUN_SIM", "0") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Import the plugin, the AutoE2E model code and the plugin's tools the way the tests do.
for path in (REPO / "Model", PLUGIN_DIR, PLUGIN_DIR / "tools"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"repo          {REPO}")
print(f"AlpaSim       {ALPASIM_ROOT}")
print(f"checkpoint    {CHECKPOINT}  (found: {CHECKPOINT.is_file()})")
print(f"rollout logs  {ROLLOUT_LOG_DIR}  (found: {ROLLOUT_LOG_DIR.is_dir()})")
print(f"device        {DEVICE}" + (f" ({torch.cuda.get_device_name()})" if DEVICE == "cuda" else ""))

## 2. The policy

The driver rebuilds AutoE2E from the `config` saved in the training checkpoint, the same reconstruction as
`Platform/pipelines/inference.py`, so the architecture always matches the weights. The released checkpoints are the
reactive BEVFormer T8 model:

- a ResNet-50 camera backbone
- BEVFormer v2 view fusion over the current frame plus seven history frames
- a map and route fusion stage
- a GRU planner that predicts 64 steps of (acceleration, curvature) at 10 Hz, 6.4 s in total

In [ ]:
from alpasim_autoe2e.plugin import load_checkpoint_model

model, config = load_checkpoint_model(CHECKPOINT, torch.device(DEVICE))
view_fusion = config["view_fusion_kwargs"]
policy_summary = {
    "view fusion": view_fusion["architecture"],
    "backbone": config["backbone"],
    "camera slots": ", ".join(config["camera_slots"]),
    "tile size": f"{view_fusion['image_size']} px, front also {view_fusion['front_image_size']} px",
    "BEV grid": f"{view_fusion['bev_h']} x {view_fusion['bev_w']}",
    "planner": config["planner_mode"],
    "route conditioning": config["enable_route_conditioning"],
    "map / route channels": f"{config['map_context_channels']} / {config['route_channels']}",
    "parameters": f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M",
}
del model  # the driver in part 4 loads its own copy
torch.cuda.empty_cache()
pd.Series(policy_summary, name=CHECKPOINT.parent.name).to_frame()

## 3. The camera rig

`driver=autoe2e_nuplan` gives the model the six nuPlan cameras. These are rectified pinhole cameras of vehicle veh-35.

AlpaSim's `extra_cameras` can only override cameras that a NuRec scene already has. So the config renders the
nuPlan cameras on six NVIDIA camera IDs, and `configs/calibration/nuplan_nurec.json` maps them back. That is why the
rear camera is called `camera_front_tele_30fov`.

The driver picks the rig profile whose cameras equal `driver.inference.use_cameras`, then orders the cameras by the
checkpoint's `camera_slots`.

In [ ]:
from alpasim_autoe2e.config import find_rig_profile

driver_cfg = yaml.safe_load((CONFIG_DIR / "autoe2e_nuplan.yaml").read_text())
runtime_cfg = yaml.safe_load((CONFIG_DIR / "autoe2e_nuplan_configs.yaml").read_text())
CAMERAS = driver_cfg["inference"]["use_cameras"]
profile = find_rig_profile(CAMERAS)
rig = profile.ordered_by_slots(config["camera_slots"])


def optical_axis_yaw(camera) -> float:
    forward = camera.camera_to_reference[:3, 2]  # camera z axis in the rig frame (x forward, y left)
    return float(np.arctan2(forward[1], forward[0]))


def horizontal_fov(camera) -> float:
    return float(2 * np.arctan(camera.image_size_wh[0] / (2 * camera.intrinsic[0, 0])))


print(f"rig profile {profile.name}: dataset {profile.dataset_name}, native front tile {profile.native_front_camera_tile}")
pd.DataFrame(
    [
        {
            "slot": camera.slot,
            "AlpaSim camera": camera.logical_id,
            "native size (w x h)": "{} x {}".format(*camera.image_size_wh),
            "x, y, z (m)": tuple(np.round(camera.camera_to_reference[:3, 3], 2)),
            "yaw (deg)": round(np.degrees(optical_axis_yaw(camera)), 1),
            "horizontal FOV (deg)": round(np.degrees(horizontal_fov(camera)), 1),
        }
        for camera in rig
    ]
)

In [ ]:
from matplotlib.patches import Wedge

fig, ax = plt.subplots(figsize=(5.5, 5.5))
for index, camera in enumerate(rig):
    x, y = camera.camera_to_reference[:2, 3]
    yaw, half_fov = np.degrees(optical_axis_yaw(camera)), np.degrees(horizontal_fov(camera)) / 2
    # Plot axes are (right, forward); a rig yaw of 0 (forward) points up, so add 90 degrees.
    ax.add_patch(Wedge((-y, x), 3.0, 90 + yaw - half_fov, 90 + yaw + half_fov, color=f"C{index}", alpha=0.3))
    label_at = (-y - 3.5 * np.sin(np.radians(yaw)), x + 3.5 * np.cos(np.radians(yaw)))
    ax.annotate(camera.slot, label_at, ha="center", va="center", fontsize=8, color=f"C{index}")
ax.add_patch(plt.Rectangle((-0.95, -1.0), 1.9, 4.6, fill=False, color="0.4"))  # rough vehicle outline
ax.set(xlabel="right (m)", ylabel="forward (m)", title="Camera frusta in the rig frame (top view)", aspect="equal")
ax.set_xlim(-6, 6)
ax.set_ylim(-5, 7)
plt.show()

## 4. One decision, offline

AlpaSim calls `AutoE2EDriver.predict(PredictionInput)` every 0.1 s of simulated time. The input carries:

- for each camera, the frames the driver asked for: `context_length: 8` with `subsample_factor: 5`, so 8 frames
  500 ms apart
- the ego pose history in AlpaSim's local frame
- speed and acceleration
- a discrete command and the latest route, both of which AutoE2E ignores today

Here we build a **synthetic** input: flat grey images, and an ego that has been driving a gentle left curve at
8 m/s. It is only for tracing the data path, so the model's answer on grey images means nothing. Parts 6–7 use a
real rollout.

In [ ]:
from alpasim_autoe2e.parser import CONTEXT_FRAMES, egomotion_from_poses, pose_trajectory
from alpasim_autoe2e.plugin import AutoE2EDriver
from alpasim_driver.models.base import CameraFrame, DriveCommand, PredictionInput
from alpasim_grpc.v0 import common_pb2

T0_US = 10_000_000
SPEED_MPS, YAW_RATE = 8.0, 0.05


def pose_at(timestamp_us: int, x: float, y: float, yaw: float) -> common_pb2.PoseAtTime:
    pose = common_pb2.PoseAtTime(timestamp_us=int(timestamp_us))
    pose.pose.vec.x, pose.pose.vec.y = x, y
    pose.pose.quat.z, pose.pose.quat.w = np.sin(yaw / 2), np.cos(yaw / 2)
    return pose


# 10 s of 10 Hz pose history on a circular arc, as the forced-ground-truth warm-up would provide.
radius = SPEED_MPS / YAW_RATE
ego_history = [
    pose_at(t, radius * np.sin(YAW_RATE * t / 1e6), radius * (1 - np.cos(YAW_RATE * t / 1e6)), YAW_RATE * t / 1e6)
    for t in range(0, T0_US + 1, 100_000)
]
frame_times = [T0_US - (CONTEXT_FRAMES - 1 - k) * 500_000 for k in range(CONTEXT_FRAMES)]
sizes = {camera["logical_id"]: (camera["height"], camera["width"]) for camera in runtime_cfg["runtime"]["simulation_config"]["cameras"]}
prediction_input = PredictionInput(
    camera_images={
        camera: [CameraFrame(t, np.full((*sizes[camera], 3), 128, np.uint8)) for t in frame_times] for camera in CAMERAS
    },
    command=DriveCommand.STRAIGHT,
    speed=SPEED_MPS,
    acceleration=0.0,
    ego_pose_history=ego_history,
    inference_seed=0,
    previous_plan=None,
    route=None,
)

driver = AutoE2EDriver(str(CHECKPOINT), CAMERAS, device=DEVICE)
parsed = driver.parser.parse(prediction_input)
print(f"t0 = {parsed.t0_us} us, initial speed {parsed.initial_speed_mps:.2f} m/s")
pd.DataFrame(
    [
        {"model input": name, "shape": tuple(value.shape), "dtype": str(value.dtype).removeprefix("torch.")}
        for name, value in parsed.model_inputs.items()
    ]
)

The parser rebuilds the packed training tensors:

- **Tiles:** the camera frames, squashed to square tiles. The front camera also enters at 1024 px.
- **Projections:** each history projection maps current-ego points into a past image.
- **Egomotion:** 64 rows of speed, acceleration, yaw rate and curvature, derived from the poses.
- **Map and route:** these get zeros with `map_valid = route_valid = False`, the model's camera-only track. AlpaSim
  passes no map, and it passes the route as waypoints, which the plugin does not rasterize yet.

Below is the egomotion the model sees before dataset normalisation.

In [ ]:
egomotion = egomotion_from_poses(pose_trajectory(ego_history), parsed.t0_us, profile.egomotion)
fig, axes = plt.subplots(1, 4, figsize=(15, 2.6), sharex=True)
for ax, values, label in zip(axes, egomotion.T, ["speed (m/s)", "acceleration (m/s²)", "yaw rate (rad/s)", "curvature (1/m)"]):
    ax.plot(np.arange(-64, 0) * 0.1, values)
    ax.set(title=label, xlabel="time before t0 (s)")
fig.tight_layout()
plt.show()

Next, the model runs, and the training rollout (`integrate_controls_torch`) turns its 64 (acceleration, curvature)
controls into waypoints. `predict()` does exactly this. We call its internal model step only to show the raw
controls, and check the result against the public call.

In [ ]:
from training.losses.control_rollout import integrate_controls_torch

start = time.perf_counter()
controls = driver._run_model(parsed).reshape(-1, 2).float().cpu()  # internal step of predict()
if DEVICE == "cuda":
    torch.cuda.synchronize()
print(f"one AutoE2E decision: {time.perf_counter() - start:.1f} s on {DEVICE}")

positions, headings, _ = integrate_controls_torch(controls[None], torch.tensor([parsed.initial_speed_mps]))
prediction = driver.predict(prediction_input)
# BEVFormer's CUDA kernels are not bit-deterministic, so the two calls agree to within a centimetre.
assert np.allclose(prediction.selected_positions[:, :2], positions[0].numpy(), atol=1e-2)
print(f"predict() returns {len(prediction.waypoint_timestamps_us)} waypoints, 0.1 s apart, anchored at t0")

steps = np.arange(1, 65) * 0.1
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
axes[0].plot(steps, controls[:, 0])
axes[0].set(title="acceleration (m/s²)", xlabel="time after t0 (s)")
axes[1].plot(steps, controls[:, 1])
axes[1].set(title="curvature (1/m)", xlabel="time after t0 (s)")
axes[2].plot(-positions[0, :, 1], positions[0, :, 0], ".-", ms=3)
axes[2].set(title="plan in the rig frame (synthetic input)", xlabel="right (m), not to scale", ylabel="forward (m)")
axes[2].set_xlim(-5, 5)
fig.tight_layout()
plt.show()

## 5. Run a closed-loop rollout (optional)

The wizard starts the AlpaSim services (runtime, renderer, driver, physics, controller) as Docker containers. It then
plays the default NuRec scene for 20 s:

1. **Warm-up (first 7 s):** the ego follows the recording, the driver is skipped, and frames and poses accumulate.
2. **Driving:** AutoE2E decides every 0.1 s.

A rollout takes about 20 minutes of wall time on one A100, so it runs in a detached `tmux` session that outlives
the notebook. Set `RUN_SIM=1` to start it from here. The defaults expect the image `alpasim-base:latest`, built as in
the plugin README; override it with `ALPASIM_BASE_IMAGE`.

In [ ]:
wizard_command = [
    "uv", "run", "--project", "src/wizard", "alpasim_wizard",
    "deploy=local", "topology=1gpu", "driver=autoe2e_nuplan",
    "driver.model.checkpoint_path=/mnt/drivers/autoe2e/checkpoint.pt",
    "defines.base_image=" + os.environ.get("ALPASIM_BASE_IMAGE", "alpasim-base:latest"),
    f"wizard.log_dir={ROLLOUT_LOG_DIR}",
    f"hydra.searchpath=[file://{ALPASIM_ROOT / 'plugins' / 'alpasim_driver' / 'alpasim_autoe2e' / 'configs'}]",
]
print(" ".join(wizard_command[:5]) + " \\\n    " + " \\\n    ".join(wizard_command[5:]))

if RUN_SIM:
    session, log = "autoe2e-closed-loop", ROLLOUT_LOG_DIR.with_suffix(".log")
    shell_command = " ".join(f"'{part}'" for part in wizard_command) + f" 2>&1 | tee -a '{log}'"
    tmux = ["tmux", "new-session", "-d", "-s", session, "-c", str(ALPASIM_ROOT)]
    tmux += ["-e", f"HF_TOKEN={os.environ['HF_TOKEN']}"] if "HF_TOKEN" in os.environ else []
    subprocess.run([*tmux, shell_command], check=True)
    print(f"started tmux session {session}, log {log}")
    while subprocess.run(["tmux", "has-session", "-t", f"={session}"], capture_output=True).returncode == 0:
        time.sleep(30)

completed = sorted(ROLLOUT_LOG_DIR.glob("rollouts/*/*/_complete"))
print(f"{len(completed)} finished rollout(s) under {ROLLOUT_LOG_DIR}")

## 6. What happened in the rollout

AlpaSim writes every rollout to `rollout.asl`, a log of protobuf messages. `tools/score_rollout.py` already reads it:

- **Recorded drive:** it takes the recorded drive from the rollout metadata.
- **Plans:** each `driver_return` message is one plan. The first pose of each plan is where the ego actually was
  at decision time, so the plans also give us the driven path.

In [ ]:
from alpasim_utils.logs import async_read_pb_log
from score_rollout import read_rollout, se3, to_trajectory

rollout_logs = sorted(ROLLOUT_LOG_DIR.glob("rollouts/*/*/rollout.asl"))
if not rollout_logs:
    raise FileNotFoundError(f"No rollout.asl under {ROLLOUT_LOG_DIR}: run part 5 (RUN_SIM=1) or the wizard first.")
rollout_asl = rollout_logs[0]

message_types = collections.Counter()
async for message in async_read_pb_log(str(rollout_asl)):
    message_types[message.WhichOneof("log_entry")] += 1
recorded, plans = await read_rollout(str(rollout_asl))

print(rollout_asl.relative_to(ROLLOUT_LOG_DIR))
print(f"{len(plans)} AutoE2E decisions; messages in the log:")
pd.Series(message_types, name="count").sort_values(ascending=False).to_frame()

In [ ]:
decision_us = np.array([int(plan[0].timestamp_us) for plan in plans])
ego_xy = np.array([[plan[0].pose.vec.x, plan[0].pose.vec.y] for plan in plans])
recorded_us = np.arange(recorded.time_range_us.start, recorded.time_range_us.stop, 100_000)
recorded_xy = np.array([se3(recorded, int(t))[:2, 3] for t in recorded_us])
recorded_at_decisions = np.array([se3(recorded, int(t))[:2, 3] for t in decision_us])
ego_deviation = np.linalg.norm(ego_xy - recorded_at_decisions, axis=1)
elapsed_s = (decision_us - decision_us[0]) / 1e6

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(recorded_xy[:, 0], recorded_xy[:, 1], color="0.6", lw=4, label="recorded drive")
for plan in plans[:: max(1, len(plans) // 8)]:
    plan_xy = np.array([[p.pose.vec.x, p.pose.vec.y] for p in plan])
    axes[0].plot(plan_xy[:, 0], plan_xy[:, 1], color="tab:orange", lw=0.8)
axes[0].plot(ego_xy[:, 0], ego_xy[:, 1], color="tab:blue", lw=2, label="ego (AutoE2E driving)")
axes[0].plot([], [], color="tab:orange", lw=0.8, label="plans, 6.4 s each")
axes[0].set(title="Top view, AlpaSim local frame", xlabel="x (m)", ylabel="y (m)", aspect="equal")
axes[0].legend()
axes[1].plot(elapsed_s, ego_deviation)
axes[1].axhline(3.0, color="tab:red", ls="--", label="3 m renderer validity bound")
axes[1].set(title="Ego distance from the recorded drive", xlabel="time since the first decision (s)", ylabel="m")
axes[1].legend()
fig.tight_layout()
plt.show()

AlpaSim also saves the front camera render as a video. Here are three frames from it. The frames are NuRec renders of
NVIDIA's dataset, so the committed copy of this notebook keeps no outputs.

In [ ]:
import cv2

videos = sorted(rollout_asl.parent.glob("*.mp4")) or sorted(rollout_asl.parent.rglob("*.mp4"))
print("videos:", [str(video.relative_to(rollout_asl.parent)) for video in videos])
if videos:
    capture = cv2.VideoCapture(str(videos[0]))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, index in zip(axes, np.linspace(0, frame_count - 1, 3).astype(int)):
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ok, frame = capture.read()
        if ok:
            ax.imshow(frame[..., ::-1])
        ax.set(title=f"{videos[0].name}: frame {index} of {frame_count}")
        ax.axis("off")
    capture.release()
    fig.tight_layout()
    plt.show()

The log also keeps every frame the driver received: one `driver_camera_image` message per camera per 0.1 s step.
These are the observations an RL update would train on. Here are the six cameras at the middle decision, in the
model's slot order.

In [ ]:
decision_t0 = int(plans[len(plans) // 2][0].timestamp_us)
latest = {}
async for message in async_read_pb_log(str(rollout_asl)):
    if message.WhichOneof("log_entry") == "driver_camera_image":
        image = message.driver_camera_image.camera_image
        if image.frame_end_us <= decision_t0 and image.frame_end_us >= latest.get(image.logical_id, (0, b""))[0]:
            latest[image.logical_id] = (image.frame_end_us, image.image_bytes)

fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, camera in zip(axes.flat, rig):
    frame_end_us, data = latest[camera.logical_id]
    ax.imshow(cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)[..., ::-1])
    ax.set_title(f"{camera.slot} ({camera.logical_id}), {(decision_t0 - frame_end_us) / 1e3:.0f} ms before t0", fontsize=9)
    ax.axis("off")
fig.tight_layout()
plt.show()

## 7. The phase-1 reward

`GroundTruthDeviationReward` (`alpasim_autoe2e/rewards.py`) scores each decision. It is the reward agreed in #123 and
#177.

- **Tracking:** `-(1.0·ADE + 0.5·FDE)` between the plan and the recorded drive at the same timestamps, in the rig
  frame at decision time.
- **Renderer validity bound:** once the ego itself is more than 3 m from the recorded drive, a −10 penalty is added
  and the decision is flagged `truncated`. Beyond that distance the Gaussian-splat reconstruction is unreliable.

`score_rollout.score()` wraps the async log reader in `asyncio.run()`. That cannot run inside the notebook's own
event loop, so it runs in a worker thread here.

In [ ]:
from alpasim_autoe2e.rewards import GroundTruthDeviationReward
from score_rollout import score

with ThreadPoolExecutor(max_workers=1) as pool:
    rows = pool.submit(score, str(rollout_asl), GroundTruthDeviationReward()).result()
scores = pd.DataFrame(rows)
scores["time (s)"] = (scores["t0_us"] - scores["t0_us"].iloc[0]) / 1e6
print(f"{len(scores)} decisions, {int(scores['truncated'].sum())} truncated, mean reward {scores['reward'].mean():.2f}")
scores[["ade_m", "fde_m", "ego_deviation_m", "tracking", "boundary", "reward"]].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
axes[0].plot(scores["time (s)"], scores["ade_m"], label="ADE")
axes[0].plot(scores["time (s)"], scores["fde_m"], label="FDE")
axes[0].set(title="Plan vs. recorded drive", xlabel="time since the first decision (s)", ylabel="m")
axes[0].legend()
axes[1].plot(scores["time (s)"], scores["reward"], color="tab:green")
truncated = scores[scores["truncated"]]
axes[1].scatter(truncated["time (s)"], truncated["reward"], color="tab:red", s=12, label="truncated")
axes[1].set(title="Phase-1 reward per decision", xlabel="time since the first decision (s)")
axes[1].legend()
fig.tight_layout()
plt.show()

## 8. From a rollout to fine-tuning data

An RL update learns from transitions. Here is the per-decision table this rollout yields. Each row has:

- when the decision was made, and where the ego was
- the plan the policy chose, as 64 waypoints in the rig frame
- its reward
- whether the episode should end there (the first truncation)

The observations, meaning the camera frames the driver saw, are in `rollout.asl` too (`driver_camera_image`, part 6).
A training loop can therefore pair each decision with its inputs without re-rendering. At about 130 MB per 20 s
rollout, though, storing them dominates the cost of logging.

In [ ]:
def plan_in_rig_frame(poses) -> np.ndarray:
    plan = to_trajectory(poses)
    rig_from_local = np.linalg.inv(se3(plan, int(poses[0].timestamp_us)))
    return np.array([(rig_from_local @ se3(plan, int(p.timestamp_us)))[:2, 3] for p in poses[1:]])


transitions = pd.DataFrame(
    {
        "decision": scores["decision"],
        "t0_us": scores["t0_us"],
        "ego_x": ego_xy[:, 0],
        "ego_y": ego_xy[:, 1],
        "ego_yaw": [2 * np.arctan2(plan[0].pose.quat.z, plan[0].pose.quat.w) for plan in plans],
        "plan_xy_rig": [plan_in_rig_frame(plan) for plan in plans],
        "reward": scores["reward"],
        "truncated": scores["truncated"],
    }
)
first_truncation = transitions.index[transitions["truncated"]].min() if transitions["truncated"].any() else None
transitions["done"] = transitions.index >= first_truncation if first_truncation is not None else False
transitions["done"] |= transitions.index == transitions.index[-1]
print(f"episode length: {int((~transitions['done']).sum()) + 1} of {len(transitions)} decisions")
transitions.head()

### Where the fine-tuning update plugs in (planned)

Everything above runs today. The loop that would turn tables like this into a better policy does not exist yet. It
is the future work in `Design/reinforcement_learning.md` §6–7, and would need:

1. **An RL environment around AlpaSim.** It ends an episode at the first `truncated` decision instead of driving to
   the end of the scene, and turns the logged frames, plans and rewards into training samples.
2. **Rollout collection over many scenes.** This is either the full pixel simulator ("Tier-P", this notebook) or
   the faster latent world-model rollouts ("Tier-L", `SimAdapter`).
3. **A policy update.** It keeps imitation learning as a regularisation term, following RAD. The phase-2 reward
   terms (safety, progress, comfort, faithfulness-gated reasoning) come from a `RewardRegistry`.
4. **Re-evaluation of the new checkpoint** in the same closed loop.

The design document does not choose an update algorithm. This notebook deliberately stops at the data such an update
would consume.